### Large datasets: public data and your own big batches

`datasets.yaml` lists the datasets. Four steps:
1. **Download** from Zenodo (checksums verified; files already here are skipped).
2. **Ingest**: every file → BDF columns (`test_time_second`, `voltage_volt`, `current_ampere`, ...), cleaned, one Parquet file per cell in `Data/Public/store/`. Runs in parallel; unchanged files are skipped on the next run.
3. **Cycles and features**: one row per cycle and one row per cell, the same for coin, pouch and cylindrical cells.
4. **Figures** in `Figures/Datasets/`.

Check the `issues` column after every ingest: it says what was cleaned or guessed (sentinel values, unsigned current, ...).

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "..")
import pandas as pd
pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)
from battery_analysis.datasets import load_spec, download, ingest, DatasetStore
from battery_analysis.datasets.plot import plot_figures
from battery_analysis.datasets.labels import cell_table
from battery_analysis.datasets.features import feature_table

spec = load_spec("datasets.yaml")
store = DatasetStore(spec["_store"])
fig_dir = spec["_project_dir"] / "Figures" / "Datasets"

### 1. Download
`dry_run=True` shows file names and sizes first. Datasets with `enabled: false` (e.g. the 2.5 GB Empa set) are only fetched when named: `download(spec, ["empa_aurora_full"])`.

In [ ]:
download(spec, dry_run=True)

In [ ]:
# download(spec)          # fetch everything enabled that is missing

### 2. Ingest

In [ ]:
report = ingest(spec)     # add force=True to redo everything
report[report["status"] != "unchanged, skipped"]

In [ ]:
cat = store.catalog()
display(store.summary())
cat[["cell_id", "form_factor", "chemistry", "test_type", "kind", "n_rows", "n_cycles", "issues"]]

### 3. Cycles and features

In [ ]:
cycles = store.cycles()
cycles.head()

In [ ]:
features = feature_table(store)
features[["cell_id", "form_factor", "n_cycles", "q0_ah", "retention_last", "eol_cycle", "fade_pct_per_100", "ce_median"]]

### 4. Publication figures
Every figure is listed in `datasets.yaml` → `figures:` (kind, cells, size, axis limits, legend), drawn in the house style and saved to `Figures/Datasets/<dataset>/<name>_<dataset>.svg`. Plot data for Origin/SigmaPlot goes to `exports/PlotData/Datasets/`. Legends show the short names from `labels:`, never file names; large sets are coloured by `group:` (here, build date).

In [ ]:
display(cell_table(store, spec)[["cell_id", "label", "group", "colour"]].head(12))
written = plot_figures(store, spec)          # or plot_figures(store, spec, ["m50_rate_profiles"])
written

### Query across everything
The store is plain Parquet, so pandas, polars or DuckDB can read it directly, e.g. every cell's capacity at cycle 100:
```python
c = store.cycles(); c[c.cycle == 100][["cell_id", "discharge_ah", "ce"]]
```

### 5. Empa: 199 coin cells (about 100 million rows)
Each cell has a metadata file (BattINFO JSON-LD). `store.metadata()` turns it into one row per cell, keeping only the fields that **differ** between cells: loadings, electrolyte, current density, build batch. Joining it to the features shows which build or test conditions go with long life.

Correlation, not cause: build batch, loading and protocol change together in this set. Use a DOE to separate them.

In [ ]:
f = feature_table(store, "empa_aurora_full")
meta = store.metadata("empa_aurora_full")
d = f.merge(meta, on="cell_id").rename(columns={
    "TestObject.dateCreated": "built",
    "TestObject.PositiveElectrode.Coating.ActiveMaterial.MassLoading [MilliGM-PER-CentiM2]": "cathode_loading_mg_cm2",
    "MeasurementParameter.Task.Next.Next.Next.Task.Input.ElectricCurrentDensity [MilliAmperePerSquareCentiMetre]": "cycling_j_mA_cm2",
})
d["batch"] = d["TestObject.productID"].astype(str).str.split(";").str[0].str.rsplit("_", n=1).str[0]
d.groupby("batch").agg(cells=("cell_id", "size"), eol_median=("eol_cycle", "median"),
                       ce_median=("ce_median", "median"), fade_median=("fade_pct_per_100", "median")).sort_values("eol_median")

In [ ]:
# cells that behave unlike the rest: check these before using the set for modelling
odd = d[(d.ce_median > 1.02) | (d.ce_median < 0.97) | (d.retention_last > 1.05)]
odd[["cell_id", "batch", "n_cycles", "q0_ah", "retention_last", "ce_median", "eol_cycle", "issues"]]

### 6. Benchmark: how long does a large dataset take?
`run()` ingests the first 10, 25, 50, 100 and all 199 Empa cells into a throw-away store with 1, 2, 4 and 6 workers, recording wall time and the time per stage (read, clean, cycle summary, write). It also times loading every cycle table, the feature table and a 199-cell figure. Results: `exports/Benchmark/` (CSV + machine.json); figures in `Figures/Datasets/benchmark/`. Takes about 5 minutes. Close other heavy programs first; times are warm-cache (raw files already read once).

In [ ]:
from battery_analysis.datasets.benchmark import run, summary, plot_benchmark
bench = run(spec, "empa_aurora_full", cells=[10, 25, 50, 100, None], workers=[1, 2, 4, 6])
display(summary(bench).round(2))
plot_benchmark(bench, spec)